# VIBES_PIPE Part 1: Data Prep + Manifest + Preprocess Demo

This notebook demonstrates that the current pipeline components work end-to-end:

- Create/verify `pairs.json` for subject folders
- Run the `prep` CLI to snapshot data into a workspace
- Inspect `manifest.json` (dataset snapshot)
- Load one subject and run preprocessing (CLAHE → resample → resize → normalize)
- Confirm output shapes and basic sanity checks

> Assumptions:
> - Your repo is at `/files/smooi/Desktop/vibes_pipe`
> - You already have a `pairs.json` under `experiments/mre_data_prep_test/`
> - Your `.mat` does NOT contain spacing, and spacing comes from the companion `.nii/.nii.gz`

In [1]:
import os
from pathlib import Path

REPO = Path("/files/smooi/Desktop/vibes_pipe").resolve()
EXP  = REPO / "experiments" / "mre_data_prep_test"
PAIRS_JSON = EXP / "pairs.json"
WORKSPACE  = EXP / "workspace_root"
MANIFEST   = WORKSPACE / "manifest.json"

print("REPO:", REPO)
print("EXP:", EXP)
print("PAIRS_JSON exists:", PAIRS_JSON.exists())
print("WORKSPACE:", WORKSPACE)
print("MANIFEST exists:", MANIFEST.exists())

# Make sure Python can import vibes_pipe without installing as a package
os.environ["PYTHONPATH"] = str(REPO / "src")
print("PYTHONPATH set to:", os.environ["PYTHONPATH"])

REPO: /files/smooi/Desktop/vibes_pipe
EXP: /files/smooi/Desktop/vibes_pipe/experiments/mre_data_prep_test
PAIRS_JSON exists: True
WORKSPACE: /files/smooi/Desktop/vibes_pipe/experiments/mre_data_prep_test/workspace_root
MANIFEST exists: True
PYTHONPATH set to: /files/smooi/Desktop/vibes_pipe/src


## A) (CLI) Generate `pairs.json` from subject folders

Pairs.json is a file path manager which records all the data's file path and data split. We can write the pairs.json manually or updated with the script - "vibes_pipe/src/vibes_pipe/data/make_pairs_from_subject_folders.py". 

Why we need this step: Your `prep` command expects a `pairs.json` file that describes **where each subject’s files live** (raw paths), so the CLI can copy them into a reproducible workspace and generate `manifest.json`.

Your raw_data folder typically contains one folder per subject, e.g.
raw_data/G009/G009_t2stack.mat, raw_data/G009/G009_mask.mat, raw_data/G009/G009_t2stack.nii, etc.

```text
raw_data/
G009/
G009_t2stack.mat
G009_t2stack.nii
G009_mask.mat
G009_Mu.mat
G039/
...

In [ ]:
# Two different modes of spliting:

# Standard build + reproducible split
# [Example] python update_pairs.py --root /data/mre --out /data/pairs.json --split auto --seed 42 --report /data/pairs_report.json
#!python update_pairs.py --root /home/smooi/Desktop/vibes_pipe/experiments/mre_data_prep_test/raw_data --out /home/smooi/Desktop/vibes_pipe/experiments/mre_data_prep_test/pairs.json --split auto --seed 42 
# Keep existing splits (useful if you later add new fields but don’t want split changes)
#[Example] python update_pairs.py --root /data/mre --split keep

# We are enforcing the splitting at the very beginning. 
# That will let us to (a) easier to reproduce result, and (b) makes the entire organization easier. 

import sys
sys.path.append("/files/smooi/Desktop/vibes_pipe/src")
!python -m vibes_pipe.data.make_pairs_from_subject_folders \
  --root /home/smooi/Desktop/vibes_pipe/experiments/mre_data_prep_test/raw_data \
  --out /home/smooi/Desktop/vibes_pipe/experiments/toast_data/pairs.json \
  --split auto \
  --seed 42 \
  --noise-root /home/smooi/Desktop/toast/noise_augmentation

In [ ]:
# Let's try with another bigger dataset
import sys
sys.path.append("/files/smooi/Desktop/vibes_pipe/src")
!python -m vibes_pipe.data.make_pairs_from_subject_folders \
  --root /home/smooi/Desktop/toast/data/toast_pipe_data \
  --out /home/smooi/Desktop/vibes_pipe/experiments/toast_data/pairs.json \
  --split auto \
  --seed 42 \
  --noise-root /home/smooi/Desktop/toast/noise_augmentation

## B) (CLI) Run `prep` to build workspace + manifest

What is `manifest.json`?

`manifest.json` is a **reproducible snapshot** of your dataset in a stable workspace layout.

It records:
- `workspace_root`: absolute folder path where the snapshot lives
- `pairs`: list of items each with:
  - `id`, `split`
  - `files`: where each file was copied (`dst` is workspace-relative)
  - `sha256`: optional hash for integrity checking
  - `meta.geometry_preprocess`: original shapes + spacing (spacing comes from X_nii)

Why it matters:
- training/inference should use **workspace files**, not raw files
- if raw files change, your experiment remains reproducible

This command:
- validates `pairs.json`
- copies files into `workspace_root/{train,val,test}/{id}/`
- writes `workspace_root/manifest.json`

Run:

In [ ]:
import sys
sys.path.append("/files/smooi/Desktop/vibes_pipe/src")
# Run the below lines in your terminal!
!python -m vibes_pipe.cli.pipeline_cli prep \
  --pairs_json experiments/mre_data_prep_test/pairs.json \
  --out_dir experiments/mre_data_prep_test/workspace_root

In [ ]:
# Let's try with another bigger dataset
import sys
sys.path.append("/files/smooi/Desktop/vibes_pipe/src")
!python -m vibes_pipe.cli.pipeline_cli prep \
  --pairs_json /home/smooi/Desktop/vibes_pipe/experiments/toast_data/pairs.json \
  --out_dir /home/smooi/Desktop/vibes_pipe/experiments/toast_data/workspace_root
  

Below, I will show you what a manifest.json looks like. 

In [ ]:
import json
from pathlib import Path

root = Path("/home/smooi/Desktop/vibes_pipe/experiments/toast_data/workspace_root")
mani_place = root / "manifest.json"

with open(mani_place, 'r') as f:
    data = json.load(f)
    
data.keys()

In [ ]:
import pandas as pd

df = pd.json_normalize(
    data['pairs'], 
    sep='_',
    record_path=None
)

# 3. Clean up column names
summary_cols = [
    'id', 
    'split', 
    'meta_geometry_preprocess_orig_t2stack_shape',
    'meta_geometry_preprocess_orig_t2stack_spacing',
    'meta_geometry_preprocess_orig_t2stack_dtype'
]

# Rename for a cleaner look
rename_map = {
    'meta_geometry_preprocess_orig_t2stack_shape': 'Shape',
    'meta_geometry_preprocess_orig_t2stack_spacing': 'Spacing',
    'meta_geometry_preprocess_orig_t2stack_dtype': 'Dtype'
}

summary_df = df[summary_cols].rename(columns=rename_map)
summary_df

# C) Run preprocessing


For the data preprocessing, we are going to run the following: 
- CLAHE (slice-wise) --> clip_limit=2.0, tile_grid_size: [8, 8], percentile_clip: [1, 99]
- resample --> [1.5, 1.5, 1.5]
- resize to `target_size` --> [128, 128, 64] 
- normalize
- label binarization (thres=0.5)

More notes about preprocessing:

1. Shape vs Spacing

- **Shape** = number of voxels in each dimension (e.g., `[160, 160, 80]`)
- **Spacing** = physical size per voxel (mm/voxel), extracted from NIfTI header

Why spacing matters:
- Subjects from different scanners can have different voxel spacing.
- If you want consistent physical resolution across subjects, you resample based on spacing.
- Masks do not need their own spacing source — they follow the T2 voxel grid.

2. Config setting
The config of preprocessing is stored in the yaml file under vibes_pipe/config/preprocess.yaml. You are able to adjust the parameters in preprocess.yaml if you wish. 


In [2]:
import numpy as np
from pathlib import Path
import yaml
from vibes_pipe.data.dataloaders import build_loaders

# Define your config location
path = Path("/home/smooi/Desktop/vibes_pipe/configs/preprocess.yaml")
cfg = yaml.safe_load(path.read_text())

# Define your manifest.json location
manifest_path = Path("/home/smooi/Desktop/vibes_pipe/experiments/mre_data_prep_test/workspace_root/manifest.json")
train_loader, val_loader, test_loader = build_loaders(manifest_path, cfg)

In [3]:
batch = next(iter(train_loader))
print(batch.keys())
print(batch["id"])
print(batch["image"].shape, batch["label"].shape)

dict_keys(['image', 'label', 'id', 'split', 'scanner_type', 'meta', 'paths'])
['G039']
torch.Size([1, 128, 128, 64]) torch.Size([1, 128, 128, 64])


Image after preprocessing: 

In [ ]:
from vibes_pipe.viz import SliceSpec, plot_image_label_slice

batch = next(iter(train_loader))
# batch["image"] shape usually [B, D, H, W] or [B, H, W, D] depending on your pipeline

i = 0  # pick sample in batch
img = batch["image"][i]
lbl = batch["label"][i]
sid = batch["id"][i]

fig = plot_image_label_slice(
    img, lbl,
    spec=SliceSpec(axis=2, index=14, rotate_k=0),
    title=f"{sid} axis=2 slice=14",
    overlay=True,
    alpha=0.3,
    show_contour=True,
)

# D) Train-time Augmentation

For the train-time augmentation, we enforce the spatial augmentation (random scaling and rotation) and noise injection. Noise profile is extracted from GE subjects and Siemens subjects and stored in a separate folder in workspace. 

The config of augmentation can be altered via augmentation.yaml.

Note: 
- Current code related to noise profile assumes that the magimg/imgraw are extracted from scans (mod needed in our matlab code if magimg or imgraw does not exist yet..)
- if noise profile is not provided and the paris.json does not store the path location of the "noise", the pipeline will just skip using the noise injection for that subject during training. 

Some tools we built: 
1. **Process Noise Profile Extraction in Batch**

**Why need this & usage senario**: When we are processing for a particular cohort, I am assuming they uses the same scanner, so the algorithm used to extract noise profiles are the same for all the subjects from that cohort. 

**Where is the script**: Inside src/vibes_pipe/augmentation/process_noise_batch.

**How to use it?** Goes into the script and changes the scanner types. run it in terminal. 

In [ ]:
# Extraction of Noise profiles in Batch 
# Below code is for magimg/imgraw

# from src.vibes_pipe.augmentation.process_noise_batch import *
# path = "/home/smooi/Desktop/vibes_pipe/experiments/toast_data/workspace_root/manifest.json"
# run_batch(path)

In [ ]:
# quick update/overwrite on manifest.json
!python src/vibes_pipe/cli/quick_update_manifest.py

# okie, finally the path looks correct now. 

In [ ]:
# # safest order
# sample or load the noise field
# apply the same spatial transform to both:
# the image/label pair
# the noise field
# inject the transformed noise into the transformed image

In [3]:
# Second, if you have already processed noise profiles, named in "*_noise.mat", you should be able to use them directly for your training 
# Attach augmentation only to train
from src.vibes_pipe.data.dataset import * 
from src.vibes_pipe.data.transforms import * # import for preprocessor 
from src.vibes_pipe.augmentation.builders import build_train_augmenter
import yaml

path_preprocess = Path("/home/smooi/Desktop/vibes_pipe/configs/preprocess.yaml")
path_augment = Path("/home/smooi/Desktop/vibes_pipe/configs/augmentation.yaml")
manifest_path = Path("/home/smooi/Desktop/vibes_pipe/experiments/toast_data/workspace_root/manifest_with_noise.json")
# load configs
cfg_preprocess = yaml.safe_load(path_preprocess.read_text())
cfg_augment = yaml.safe_load(path_augment.read_text())

# build objects
preprocessor = Preprocessor(cfg_preprocess)
train_augmenter = build_train_augmenter(cfg_augment)

train_ds = ManifestDataset(
    manifest=manifest_path,
    split="train",
    preprocessor=preprocessor,
    augmenter=train_augmenter,
)

val_ds = ManifestDataset(
    manifest=manifest_path,
    split="val",
    preprocessor=preprocessor,
    augmenter=None,
)

test_ds = ManifestDataset(
    manifest=manifest_path,
    split="test",
    preprocessor=preprocessor,
    augmenter=None,
)

In [ ]:
# Pytorch training normally uses:
# Dataset --> Dataloader --> Model 



# E) Build Model
We can now freely construct our model with the yaml file to build the model. Below, we're going to test the model builder function. 

In [1]:
import yaml
from src.vibes_pipe.models.builders import build_model

with open("/home/smooi/Desktop/vibes_pipe/configs/probabilistic_train.yaml", "r") as f:
    cfg = yaml.safe_load(f)

model = build_model(cfg)

print(model)
print(type(model))

Probabilistic U-Net with latent vector injection enabled.PriorNet and PosteriorNet will be used.
ProbUNet3D(
  (unet): UNet(
    (model): Sequential(
      (0): ResidualUnit(
        (conv): Sequential(
          (unit0): Convolution(
            (conv): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
            (adn): ADN(
              (N): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
              (D): Dropout(p=0.3, inplace=False)
              (A): PReLU(num_parameters=1)
            )
          )
          (unit1): Convolution(
            (conv): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (adn): ADN(
              (N): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
              (D): Dropout(p=0.3, inplace=False)
              (A): PReLU(num_parameters=1)
            )
          )
        )
        (residual): Conv3d(1, 16, kernel_s

/home/smooi/miniconda3/envs/torch3090/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# F) Training with Training Engines

In [ ]:
# load config
import yaml
with open("/home/smooi/Desktop/vibes_pipe/configs/deterministic_train.yaml", "r") as f:
    cfg = yaml.safe_load(f)
    